# Skills Catalog MCP — Context-Only Skill Discovery

**Status:** Proposed  
**Date:** 2026-07-31  
**Scope:** `spur-core`, `spur-mcp`, `spur-tui`, agent runtime projection

## Decision

SPUR will give coding agents exactly one bootstrap skill, `skills_catalog`. Agents discover and load task-specific skills through a read-only MCP interface backed by the same catalog used by the Explore TUI. Retrieved skill instructions and resources are returned into the current conversation context; they are never materialized into the worker's `.skills/` directory.

The v1 serving interface has two tools:

- `skill_search` discovers a small ranked set of eligible skills.
- `skill_read` loads an exact `SKILL.md` or one referenced resource.

The Explore TUI remains the human governance plane for synchronization, inspection, gating, adoption, and removal. The MCP interface is the agent serving plane and exposes only enabled, compatible bundled skills or enabled, compatible adopted-and-gate-approved skills.

## Why this decision

Preloading a large skill collection consumes context and increases incorrect skill selection. Industry systems increasingly use progressive disclosure: keep a small bootstrap instruction in context, retrieve candidates on demand, and load full instructions only after selection. Research also identifies retrieval accuracy and skill shadowing—not merely raw context size—as the principal risks when skill libraries grow.

This specification therefore optimizes for:

1. minimal startup context;
2. explicit, observable skill selection;
3. one authoritative catalog and trust policy;
4. exact versioned reads without filesystem mutation;
5. repeated discovery as the task evolves.

In [ ]:
flowchart LR
    CONTRACT["`@spec SKILLS-CATALOG-ARCHITECTURE
@type DeliveryOutcome = enum[returned, blocked]
@type WriteEffect = enum[none, written]
@input eligible: Bool
@input read_requested: Bool
@output delivery: DeliveryOutcome
@output write_effect: WriteEffect`"]

    RETURN["`@branch RETURN
@when eligible and read_requested
@ensures RETURN_RESULT: delivery = returned
@ensures RETURN_NO_WRITE: write_effect = none`"]

    BLOCK["`@branch BLOCK
@when not (eligible and read_requested)
@ensures BLOCK_RESULT: delivery = blocked
@ensures BLOCK_NO_WRITE: write_effect = none`"]

    CHECK["`@verify ARCH_DETERMINISTIC: prove determinism
@verify ARCH_COVERAGE: prove partition_coverage
@verify ARCH_EXCLUSIVE: prove partition_exclusive
@verify RETURN_REACHABLE: witness branch RETURN
@verify BLOCK_REACHABLE: witness branch BLOCK`"]

    subgraph Human[Human governance plane]
        TUI[Explore TUI]
        Sync[Sync and inspect]
        Gate[Gate and adopt]
        TUI --> Sync --> Gate
    end

    subgraph Catalog[Authoritative catalog and pool]
        Entries[Catalog entries]
        Policy[Eligibility policy]
        Content[Pinned skill content]
        Entries --> Policy
        Content --> Policy
    end

    subgraph Runtime[Agent serving plane]
        Bootstrap[skills_catalog bootstrap skill]
        Search[skill_search]
        Read[skill_read]
        Context[Conversation context]
        Bootstrap --> Search
        Search -->|opaque skill reference| Read
        Read -->|verified text| Context
    end

    CONTRACT --> RETURN --> CHECK
    CONTRACT --> BLOCK --> CHECK
    Gate --> Catalog
    Catalog --> Search
    Catalog --> Read
    Worker[Coding agent] --> Bootstrap
    Context --> Worker
    Denied[Synced but unapproved skills] -. Explore only .-> TUI
    Denied -. blocked .-> Search
    Denied -. blocked .-> Read
    Read -. write effect is none .-> FS[Worker .skills directory]

## Goals and non-goals

### Goals

- Project only `skills_catalog` into each coding agent runtime.
- Search the eligible skill pool by task intent without exposing every skill body or metadata record at startup.
- Load selected instructions directly into conversation context with immutable provenance.
- Reuse the Explore catalog, pinning, content hashes, and governance state.
- Enforce authorization again at read time so stale search results cannot bypass policy.
- Support repeated discovery and lazy loading of referenced files.
- Make retrieval measurable through offline evaluations and privacy-preserving runtime telemetry.
- Provide a migration and rollback path from the current all-active-skills projection.

### Non-goals

- Installing skills from the agent runtime.
- Allowing an agent to adopt, approve, update, or remove catalog entries.
- Exposing synced-but-unapproved skill content to coding agents.
- Building an embedding service or autonomous hierarchical retriever in v1.
- Executing packaged scripts or serving binary assets from retrieved skills; v1 serves verified text only.
- Defining a universal dependency solver for skill composition.
- Replacing the Explore TUI's human-oriented catalog management workflow.

## Normative invariants

1. **Single bootstrap:** a coding-agent projection contains `skills_catalog` and no task-specific catalog skills.
2. **Context only:** successful reads return content in the MCP result and perform no worker-filesystem writes.
3. **Eligibility on every operation:** both search and read require `enabled ∧ compatible ∧ (bundled ∨ (adopted ∧ gate-approved))`; bundled skills are approved through the SPUR release process.
4. **Pinned identity:** a read resolves the exact source revision and content hash named by the search result.
5. **Fail closed:** missing, stale, removed, unapproved, incompatible, or hash-mismatched skills are not returned.
6. **Authority preservation:** retrieved instructions cannot override system, developer, user, repository, or project-management authority.
7. **Bounded disclosure:** search returns summaries; full instructions and resources require an explicit read.
8. **No raw-query logging by default:** production telemetry records counts, latency, result identifiers, ranks, hashes, and outcomes—not source code or user query text.

The context-only and approval-gating invariants were encoded with `spurpower-solve`. Asserting either an unapproved successful read or a filesystem write produced `unsat`, proving that the policy rules are mutually consistent and exclude those prohibited states. This proof applies to the encoded contract; implementation tests must still establish that the code conforms to it.

In [ ]:
flowchart TD
    CTX["`@spec SKILL-ELIGIBILITY-POLICY
@type Eligibility = enum[eligible, ineligible]
@input bundled: Bool
@input adopted: Bool
@input gate_approved: Bool
@input enabled: Bool
@input compatible: Bool
@output eligibility: Eligibility`"]

    ELIGIBLE["`@branch ELIGIBLE
@when enabled and compatible and (bundled or (adopted and gate_approved))
@ensures ELIGIBLE_RESULT: eligibility = eligible`"]

    INELIGIBLE["`@branch INELIGIBLE
@when not (enabled and compatible and (bundled or (adopted and gate_approved)))
@ensures INELIGIBLE_RESULT: eligibility = ineligible`"]

    CHECK["`@verify ELIGIBILITY_DETERMINISTIC: prove determinism
@verify ELIGIBILITY_COVERAGE: prove partition_coverage
@verify ELIGIBILITY_EXCLUSIVE: prove partition_exclusive
@verify ELIGIBLE_REACHABLE: witness branch ELIGIBLE
@verify INELIGIBLE_REACHABLE: witness branch INELIGIBLE`"]

    SYNC[Synced external: Explore only] --> REVIEW[Inspect and gate]
    REVIEW -->|adopt and approve| CTX
    BUNDLE[Bundled: release-approved] --> CTX
    CTX --> ELIGIBLE --> CHECK
    CTX --> INELIGIBLE --> CHECK
    ELIGIBLE --> SERVE[Agent search and read]
    INELIGIBLE --> HIDE[Hidden from agent MCP]

## Catalog model and ownership

`spur-core::explore` remains the source of truth. The existing catalog identity fields—`source`, `rel_path`, `pinned_commit`, and `content_sha256`—form the authoritative versioned identity. MCP-facing identifiers are opaque references derived from that identity; callers must not parse or construct them.

A searchable record contains:

| Field | Purpose |
|---|---|
| `skill_id` | Opaque version-pinned reference used by `skill_read` |
| `name` | Canonical skill name |
| `description` | Concise trigger-oriented description |
| `source` | Provenance label suitable for display |
| `pinned_commit` | Immutable source revision when applicable |
| `content_sha256` | Exact content integrity check |
| `catalog_revision` | Deterministic digest of the sorted eligible catalog and policy state |
| `resource_manifest_sha256` | Integrity reference for the approved text-resource inventory |
| `compatibility` | Server-derived context-only and runtime-capability decision |
| `availability` | Agent-visible records are always `eligible`; other states never leave the governance plane |
| `match` | Rank and a concise explanation of why the record matched |

The serving index contains only eligible records. Governance state remains outside searchable instruction text so an external skill cannot influence its own approval status. `catalog_revision` is content-addressed rather than a process-local counter, allowing TUI and MCP processes to compare the same merged view. Unrelated catalog changes do not invalidate a version-pinned `skill_id`; read-time eligibility remains authoritative.

At adoption or bundle build time, the gate derives an immutable text-resource inventory containing normalized relative path, media type, size, and SHA-256 for every readable resource. This inventory is not returned by search. Skills that require executable scripts, binary assets, undeclared runtime tools, or other unsupported delivery capabilities are marked incompatible with the context-only runtime.

### Ownership boundaries

- `spur-core::explore`: catalog loading, source pinning, merged local/global view, eligibility policy, and canonical content access.
- `spur-core::mcp::skills_catalog`: a `spur_mcp::ToolModule` that binds Explore application logic to JSON schemas, authorization, stable errors, and telemetry without creating a `spur-mcp → spur-core` dependency cycle.
- `spur-mcp`: transport-neutral `ToolModule` and `ToolRegistry` contracts used by the module.
- `spur-tui::views::explore`: human sync, filtering, preview, gate results, adoption, and removal.
- Agent runtime projection: materializes only the bundled `skills_catalog/SKILL.md` bootstrap.

The catalog API should be shared Rust application logic called by both frontends. The MCP module must not scrape TUI state, and the TUI must not call MCP internally.

## MCP tool contracts

### `skill_search`

Searches only the current eligible index. It never returns instruction bodies.

**Input**

```json
{
  "query": "validate authentication changes before merging",
  "limit": 5,
  "source": null
}
```

- `query` is required, non-empty task intent expressed in natural language.
- `limit` is optional, defaults to 5, and accepts 1–5. Five favors recall while remaining inside the research-supported small candidate range; evaluations may revise this contract.
- `source` is an optional exact provenance filter.

**Output**

```json
{
  "catalog_revision": "sha256:catalog-view-digest",
  "results": [
    {
      "skill_id": "opaque-versioned-reference",
      "name": "code-change-verification",
      "description": "Verify code changes before completion...",
      "source": "bundled",
      "pinned_commit": null,
      "content_sha256": "...",
      "rank": 1,
      "match_reason": "Matched validation, code change, and pre-merge intent"
    }
  ]
}
```

V1 ranking is deterministic lexical retrieval over normalized name, description, and approved trigger metadata. Exact-name and exact-token matches receive deterministic preference; BM25 supplies relevance ranking; stable identity breaks ties. Agents may refine the query and search repeatedly. Semantic reranking is deferred until an evaluation demonstrates a recall gap.

### `skill_read`

Reads one exact skill document or one resource from the version selected by search.

**Input**

```json
{
  "skill_id": "opaque-versioned-reference",
  "resource": null
}
```

- Omit `resource` to return `SKILL.md`.
- Set `resource` to a relative path present in the approved text-resource inventory.
- V1 returns UTF-8 text resources only; it does not execute scripts or serve binary assets.
- Absolute paths, traversal segments, symlink escapes, undeclared paths, and cross-skill reads are rejected.

**Output**

```json
{
  "skill_id": "opaque-versioned-reference",
  "name": "code-change-verification",
  "source": "bundled",
  "catalog_revision": "sha256:catalog-view-digest",
  "content_sha256": "...",
  "resource": "SKILL.md",
  "media_type": "text/markdown",
  "content": "...exact verified content..."
}
```

Before returning content, `skill_read` checks current eligibility and context-only compatibility, verifies the pinned source revision and content hash, validates the requested resource against the approved inventory, and applies the catalog's content-size policy. The catalog revision is returned for observability; unrelated catalog changes do not revoke a still-eligible version-pinned reference. A search result is not an authorization capability.

### Stable error kinds

| Error | Meaning | Agent action |
|---|---|---|
| `invalid_query` | Empty or structurally invalid search | Refine the request |
| `skill_not_found` | Unknown opaque reference | Search again |
| `skill_not_eligible` | Removed, disabled, unapproved, or incompatible | Search again; do not bypass |
| `stale_skill_ref` | Target identity changed or is no longer current | Search again |
| `resource_not_found` | Resource is absent or not declared | Read the main skill or another declared resource |
| `resource_denied` | Unsafe path or unsupported media type | Stop; do not construct filesystem paths |
| `content_too_large` | Content violates catalog gate policy | Report the catalog problem |
| `integrity_mismatch` | Pinned content hash did not verify | Fail closed and report |

In [ ]:
sequenceDiagram
    participant Agent
    participant Bootstrap as skills_catalog
    participant Search as skill_search
    participant Catalog
    participant Read as skill_read

    Note over Agent,Read: @spec SKILL-READ-DECISION<br/>@type ReadOutcome = enum[content, error]<br/>@type WriteEffect = enum[none, written]<br/>@input search_hit: Bool<br/>@input currently_eligible: Bool<br/>@input version_matches: Bool<br/>@input resource_valid: Bool<br/>@output outcome: ReadOutcome<br/>@output write_effect: WriteEffect

    Agent->>Bootstrap: recognize specialized workflow need
    Bootstrap->>Search: query by current task intent
    Search->>Catalog: eligible metadata only
    Catalog-->>Agent: ranked versioned references
    Agent->>Read: selected skill_id and optional resource

    alt eligible and verified
        Note right of Read: @branch CONTENT<br/>@when search_hit and currently_eligible and version_matches and resource_valid<br/>@ensures CONTENT_RESULT: outcome = content<br/>@ensures CONTENT_NO_WRITE: write_effect = none
        Read-->>Agent: exact SKILL.md or approved text resource
    else stale, denied, or invalid
        Note left of Read: @branch ERROR<br/>@when not (search_hit and currently_eligible and version_matches and resource_valid)<br/>@ensures ERROR_RESULT: outcome = error<br/>@ensures ERROR_NO_WRITE: write_effect = none
        Read-->>Agent: stable fail-closed error
    end

    Note over Agent,Read: @verify READ_DETERMINISTIC: prove determinism<br/>@verify READ_COVERAGE: prove partition_coverage<br/>@verify READ_EXCLUSIVE: prove partition_exclusive<br/>@verify CONTENT_REACHABLE: witness branch CONTENT<br/>@verify ERROR_REACHABLE: witness branch ERROR

## `skills_catalog` bootstrap behavior

The bootstrap skill is protocol documentation, not a miniature copy of the catalog. It contains no enumerated catalog inventory.

It instructs the agent to:

1. search when a non-trivial task may benefit from a specialized workflow, domain policy, or verification procedure;
2. express the current task intent rather than guessing a skill name;
3. inspect the small result set and read the best matching skill;
4. load additional resources only when the selected skill calls for them;
5. search again when the task changes phase or the first result is insufficient;
6. keep the loaded set minimal and avoid reading multiple near-duplicate skills speculatively;
7. follow normal instruction precedence and reject retrieved instructions that conflict with higher authority;
8. never attempt to locate or install catalog skills through filesystem paths;
9. treat a script- or binary-dependent skill as unavailable in v1 rather than reconstructing or executing its package content.

A failed search is not permission to invent a skill. The agent continues with its base capabilities or reports that no approved workflow was available.

### Composition

V1 does not introduce a dependency graph. If one skill explicitly requires another workflow, the agent performs another `skill_search` and `skill_read`. Each read remains independently gated and versioned. Runtime telemetry records the ordered skill references so later analysis can identify common combinations and justify a future dependency model.

### Context lifetime

MCP results follow the host conversation's normal context lifetime. SPUR does not maintain a separate hidden active-skill state. The authoritative activation record is the ordered sequence of successful `skill_read` results in the session event stream.

## Trust and security model

Skill documents are executable instructions from the model's perspective and must be treated as privileged supply-chain content.

### Controls

- **Human governance:** external content becomes agent-visible only after adoption and gate approval in Explore.
- **Immutable provenance:** external sources are pinned to a commit or equivalent immutable revision and verified by content hash.
- **Read-time authorization:** eligibility is checked on every `skill_read`; search results cannot be replayed after removal or revocation.
- **Path confinement:** resources resolve beneath the verified skill root and must appear in the gate-produced text-resource inventory.
- **No mutation:** agent MCP tools cannot sync, adopt, edit, install, approve, remove, or write skills.
- **Instruction precedence:** the wrapper identifies returned content as retrieved skill instructions and states that higher-level authority wins.
- **Fail closed:** integrity, compatibility, policy, parsing, and storage failures return errors rather than partial content.
- **Auditability:** successful reads record skill identity, version, rank, catalog revision, and outcome without logging skill bodies or raw queries by default.

### Threats addressed

| Threat | Mitigation |
|---|---|
| Prompt injection in an arbitrary ecosystem skill | Human adoption, gate approval, eligible-only serving |
| Catalog changes between search and read | Versioned opaque reference plus read-time revalidation |
| Path traversal or cross-package resource read | Approved resource inventory, canonical containment, symlink rejection |
| Skill silently changes upstream | Pinned revision and content hash |
| Agent self-installs a discovered skill | Read-only MCP surface and no filesystem path exposure |
| Disabled skill remains in an index cache | Catalog-revision invalidation and authorization on read |
| Skill requires packaged execution or unavailable tools | Context-only compatibility gate excludes it from the serving index |
| Skill overrides repository or user intent | Explicit authority boundary in bootstrap and read envelope |

### Deliberate boundary

The agent-facing MCP search does not advertise unapproved external skills with a warning. Showing unavailable candidates would waste selection budget and encourage attempts to bypass governance. Discovery of such ecosystem content belongs to the human Explore TUI.

## Retrieval evaluation and observability

The solver establishes contract consistency, not retrieval quality. V1 must ship with a representative evaluation fixture before becoming the default projection mode.

### Offline evaluation set

Each case contains:

- a realistic coding-agent task description;
- zero, one, or several acceptable skill identities;
- prohibited or confusing near-neighbor skills;
- expected behavior when no eligible skill applies;
- catalog revision and exact skill versions.

The set must cover exact-name queries, conceptual queries, vocabulary mismatch, near-duplicate descriptions, multi-phase tasks, revoked skills, and no-match cases.

### Required measurements

| Metric | Question answered |
|---|---|
| Recall@5 | Does the candidate set contain an acceptable skill? |
| Mean reciprocal rank | How early does the first acceptable skill appear? |
| Activation precision | Does the agent read an appropriate result rather than a shadowing neighbor? |
| No-match precision | Does the system avoid recommending irrelevant skills? |
| Downstream task pass rate | Does retrieval improve actual coding outcomes? |
| Startup and cumulative skill tokens | Does the design reduce context use over a full task? |
| Search/read latency | Is repeated discovery operationally affordable? |
| Stale/denied read rate | Is catalog churn creating runtime friction? |
| Skills read per task | Are agents accumulating unnecessary instruction context? |

### Release gate

The legacy all-active-skills projection remains available until the context-only path demonstrates:

- no security or integrity invariant violations;
- retrieval quality at least as good as a frozen baseline on the agreed evaluation set;
- lower startup skill-token consumption;
- no statistically meaningful regression in downstream task pass rate;
- stable error handling under catalog updates and revocations.

Numeric pass thresholds are not invented in this specification. They must be established from a recorded baseline and approved with the implementation plan.

### Runtime events

Emit structured events for `skill_search_started`, `skill_search_completed`, `skill_read_completed`, and `skill_read_failed`. Include duration, result count, selected opaque identity, rank, catalog revision, content hash, and stable error kind. Raw queries and returned content remain excluded unless an explicit diagnostic mode is enabled.

In [ ]:
flowchart LR
    P0[Phase 0: baseline] --> P1[Phase 1: shadow retrieval] --> P2[Phase 2: opt-in catalog-only] --> CTX

    CTX["`@spec CATALOG-ROLLOUT-GATE
@type RolloutDecision = enum[retire_legacy, keep_legacy]
@input retrieval_gate_pass: Bool
@input security_gate_pass: Bool
@input integration_gate_pass: Bool
@input observation_gate_pass: Bool
@output decision: RolloutDecision`"]

    RETIRE["`@branch RETIRE
@when retrieval_gate_pass and security_gate_pass and integration_gate_pass and observation_gate_pass
@ensures RETIRE_RESULT: decision = retire_legacy`"]

    KEEP["`@branch KEEP
@when not (retrieval_gate_pass and security_gate_pass and integration_gate_pass and observation_gate_pass)
@ensures KEEP_RESULT: decision = keep_legacy`"]

    CHECK["`@verify ROLLOUT_DETERMINISTIC: prove determinism
@verify ROLLOUT_COVERAGE: prove partition_coverage
@verify ROLLOUT_EXCLUSIVE: prove partition_exclusive
@verify RETIRE_REACHABLE: witness branch RETIRE
@verify KEEP_REACHABLE: witness branch KEEP`"]

    CTX --> RETIRE --> CHECK
    CTX --> KEEP --> CHECK
    RETIRE --> P3[Phase 3: catalog-only default]
    P3 --> P4[Phase 4: retire all-skills projection]
    KEEP --> ROLLBACK[Keep or restore legacy projection]
    ROLLBACK --> P1

## Migration and implementation seams

### Phased migration

1. **Baseline:** measure current startup skill tokens, task success, and projection behavior.
2. **Shadow index:** build the eligible index and run retrieval evaluations without changing worker projections.
3. **Opt-in runtime:** add a configuration switch that projects only `skills_catalog` and exposes the two MCP tools.
4. **Default runtime:** switch new coding-agent sessions after release gates pass; preserve the legacy path for rollback.
5. **Retirement:** remove all-active-skills projection only after the observation window and compatibility review.

Rollback changes projection mode for new sessions. Existing conversations retain the context already delivered to them.

### Expected code seams

- Extend `spur-core::explore` with an agent-serving catalog facade rather than duplicating catalog loading in MCP.
- Add `spur-core::mcp::skills_catalog::SkillsCatalogMcpModule`, implementing the public `spur_mcp::ToolModule` contract. Keeping the adapter in `spur-core` avoids the existing `spur-core → spur-mcp` dependency becoming a cycle.
- Compose that module into the brain and worker registries that already receive repository-scoped tools.
- Change runtime skill projection to select `skills_catalog` when catalog-only mode is enabled.
- Keep Explore TUI behavior intact, adding only agent-eligibility visibility if existing status presentation is insufficient.
- Derive a deterministic catalog revision from the local/global merged eligible view.

No new crate or reversed dependency is required for v1 lexical retrieval. If later frontends need the serving facade without depending on `spur-core`, extracting a dedicated lower-level skills-catalog crate is a separate, measured refactor.

## Verification strategy

### Unit tests

- Eligibility filtering for bundled, adopted, rejected, disabled, removed, and incompatible entries.
- Deterministic ranking and tie-breaking.
- Opaque reference round trip and stale-reference rejection.
- Commit/hash verification and revision-based cache invalidation.
- Context-only compatibility classification for text-only, script-dependent, binary-resource, and missing-tool skills.
- Resource path traversal, absolute path, symlink escape, undeclared file, unsupported media, and cross-skill denial.
- Stable errors and response serialization.
- Bootstrap projection contains exactly one skill.

### Integration tests

- Agent searches, reads `SKILL.md`, reads one declared resource, and completes without filesystem materialization.
- Search result becomes revoked before read and fails closed.
- Local/global catalog merge cannot expose an unapproved shadowing entry.
- Two concurrent catalog revisions do not return mismatched content.
- MCP unavailable path degrades to base-agent behavior without inventing or installing skills.
- Legacy projection rollback remains functional during migration.

### Acceptance criteria

- The worker's projected `.skills/` tree contains only `skills_catalog` in catalog-only mode.
- `skill_search` returns only eligible metadata and at most five results.
- `skill_read` returns exact verified content and never mutates the worker filesystem.
- Unapproved or stale references fail even if obtained before a catalog transition.
- Explore and MCP compute the same catalog revision and eligibility decisions.
- Retrieval and downstream evaluation gates pass before catalog-only mode becomes the default.

## Evidence and alternatives

### Industry and standards

- [Anthropic Tool Search](https://platform.claude.com/docs/en/agents-and-tools/tool-use/tool-search-tool) documents on-demand discovery for large tool collections and reports substantial context reduction from loading a small relevant subset.
- [Agent Skills specification](https://agentskills.io/specification) defines progressive disclosure across metadata, `SKILL.md`, and resources.
- [Agent Skills client implementation](https://agentskills.io/client-implementation/adding-skills-support) recommends a dedicated activation tool that returns selected instructions into conversation context.
- [OpenAI Agents SDK skills case study](https://developers.openai.com/blog/skills-agents-sdk) describes repository skills used for recurring coding, verification, documentation, and release workflows.

### Research

- [More Skills, Worse Agents?](https://arxiv.org/abs/2605.24050) identifies skill shadowing and reports degradation as skill libraries expand.
- [ToolRet](https://aclanthology.org/2025.findings-acl.1258/) shows that generic information-retrieval quality does not automatically translate into reliable tool retrieval.
- [Dynamic Tool Dependency Retrieval](https://aclanthology.org/2026.findings-acl.1680/) supports repeated retrieval conditioned on an evolving task plan.
- [CODESKILL](https://arxiv.org/abs/2605.25430) reports coding improvements from a compact, dynamically retrieved procedural skill bank.
- [EASYTOOL](https://aclanthology.org/2025.naacl-long.44/) supports concise normalized discovery records with detailed instructions loaded separately.

Recent skill-specific papers include preprints; peer-reviewed tool-retrieval papers provide the more mature adjacent evidence. Production success reports are observational rather than controlled causal experiments.

### Alternatives rejected for v1

1. **Project every active skill.** Rejected because startup context and skill-shadowing risk scale with the catalog.
2. **Expose all synced external skills with warnings.** Rejected because warnings do not provide an enforceable trust boundary and unavailable results consume selection budget.
3. **Hybrid lexical/vector retrieval immediately.** Deferred because it adds index lifecycle and embedding infrastructure before a measured lexical recall gap exists.
4. **Hierarchical agentic retriever.** Deferred because it adds recursive planning, calls, latency, and failure modes disproportionate to the initial catalog.
5. **One polymorphic MCP tool.** Rejected because discovery and privileged content retrieval require distinct schemas, authorization points, telemetry, and cache behavior.
6. **Three search/read/resource tools.** Rejected because a versioned `skill_read` operation can safely distinguish the main document from one declared resource without a separate endpoint.

## Resolved decisions

- Context delivery only; no task-specific skill materialization.
- Two-tool serving surface: `skill_search` and `skill_read`.
- Eligible-only agent index; unapproved ecosystem discovery remains in Explore.
- Deterministic lexical retrieval for v1, with repeated search allowed.
- Shared `spur-core::explore` catalog logic for TUI and MCP.
- Versioned opaque references and authorization on every read.
- Evaluation-derived release thresholds rather than invented numeric quality targets.

## Executable NS-Mermaid evidence

The diagrams were authored against the current `spur-notebook` implementation rather than raw Mermaid assumptions:

- `src/ns_spec/parse.rs` defines supported flowchart, state-diagram, and sequence-diagram carriers.
- `src/ns_spec/obligations.rs::validate_constructive_totality` requires every output to have an explicit equality definition globally or in every branch.
- `src/ns_spec/tests.rs::SEQUENCE_HANDSHAKE` is the canonical participant-note and `alt`/`else` sequence fixture used by the request-flow diagram.

All four native `ns_mermaid` cells executed through Notebook MCP and emitted schema-v2 proof reports:

| Cell | Obligations | Result | Report hash |
|---|---:|---|---|
| Architecture and context-only delivery | 5/5 matched | `verified=true` | `bcf1a8f905a3b125a5f1d87a3de2da9187cf4bbde1509d5a839494ea566faa20` |
| Eligibility policy | 5/5 matched | `verified=true` | `680e7574ba00239081f67fb8bc6cd11cabacf926921c264dcd0f5dfdb1cfc60d` |
| Search/read sequence | 5/5 matched | `verified=true` | `90842fabcb998497f4bbe0f766d729e7da4aca6a1a84924f9f578e82e3c7fdbe` |
| Rollout gate | 5/5 matched | `verified=true` | `d6b1e57cf466b9750c6ecfbf22ac513890c002e4d1e06e237977e5c5a8913082` |

Each decision relation proves determinism, partition coverage, and partition exclusivity, then produces witnesses for both branches. These proofs validate the formal policy encoded in the diagrams; they do not replace Rust implementation tests or retrieval-quality evaluation.